# CNN + Frequency-Based Explainable DeepFake Detection
## Free Colab training and image analysis

This notebook contains the project source. **No trained DeepFake detector or dataset is included.**
The target is 95% accuracy, but only the evaluation cell can establish actual results.

**First-time use:** choose Runtime → Change runtime type → T4 GPU. Run one cell at a time, starting below.
**Returning after a disconnect:** run steps 1–3 to restore the project and cached data, then continue at step 7.
You lose at most the unfinished epoch when disconnected.

**Dataset access:** request FaceForensics++ using the [official instructions](https://github.com/ondyari/FaceForensics/blob/master/dataset/README.md).
Use your own details and follow the dataset agreement. Approval may take time. While waiting, you can run steps 1–3 and the software checks. Stop before step 4.

This is a ConvNeXt-Tiny adaptation of MH-FFNet with patch/global DCT enhancement. It is not an exact reproduction of the authors' ConvNeXt-Base experiment.
No paid API, subscription or public hosting is required. Free GPU availability is not guaranteed.


## 1. Unpack the included project and install free libraries
The long encoded line contains your project files. You do not need to edit it.
If Colab explicitly asks to restart the session after installing libraries, restart and rerun this cell before continuing.


In [ ]:
import base64, hashlib, io, os, sys, zipfile, subprocess
from pathlib import Path
payload = '...'
archive_bytes = base64.b64decode(payload)
assert hashlib.sha256(archive_bytes).hexdigest() == '698010a15e294316122720c64abf73c63c7c490c41ebe923ed2669d2a63b780d', 'Project archive is damaged'
ROOT = Path('/content/deepfake_project')
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
    for member in z.infolist():
        target = (ROOT/member.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project installed:', ROOT)


## 2. Check GPU and software
The software tests use synthetic inputs to check the implementation. Passing these tests does **not** establish detection accuracy.
If the GPU check says unavailable, save the notebook and try free GPU access later. Do not purchase anything.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Unavailable — select a GPU runtime or try later')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
print('Software checks passed. No detector has been trained by these checks.')


## 3. Connect Drive and restore saved data
Approve Google's Drive dialog yourself. The project creates a `CNN_Frequency_Project` folder in your Drive.
It stores cropped-data archives and checkpoints there. Keep enough free space (often 1–2 GB for the starter subset).
Your password is not needed in this notebook.

On a fresh runtime this cell restores the data archive. Once data has been restored, **skip steps 4–6** and continue at step 7.


In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
SAVE = Path('/content/drive/MyDrive/CNN_Frequency_Project')
SAVE.mkdir(parents=True, exist_ok=True)
RUN = SAVE/'runs'/'frequency'
RUN.mkdir(parents=True, exist_ok=True)
CACHE = SAVE/'face_data.zip'
if CACHE.exists() and not (ROOT/'data'/'manifest.csv').exists():
    print('Restoring prepared faces from Drive...')
    shutil.copy2(CACHE, '/content/face_data_restore.zip')
    target_root = (ROOT/'data').resolve()
    with zipfile.ZipFile('/content/face_data_restore.zip') as z:
        if any(not (target_root/x.filename).resolve().is_relative_to(target_root) for x in z.infolist()):
            raise ValueError('Unsafe cache path')
        z.extractall(target_root)
print('Data ready:', (ROOT/'data'/'manifest.csv').exists())
print('Resume checkpoint available:', (RUN/'last.pt').exists())
print('Saved work:', SAVE)


## 4. Upload the official dataset downloader — only after approval
Run this cell and select the `.py` downloader you received from the FaceForensics++ team.
Do not upload passwords, browser cookies or personal documents. The downloader itself is not included in project exports.
If you already restored `data/manifest.csv`, skip to step 7.


In [ ]:
from google.colab import files
uploaded = files.upload()
scripts = [name for name in uploaded if name.lower().endswith('.py')]
if len(scripts) != 1:
    raise ValueError('Upload exactly one official FaceForensics++ Python downloader.')
DOWNLOADER = ROOT/Path(scripts[0]).name
DOWNLOADER.write_bytes(uploaded[scripts[0]])
subprocess.run([sys.executable, str(DOWNLOADER), '-h'], check=True)


## 5. Download a free starter subset
Read the agreement accompanying your approved dataset download. Change `I_ACCEPT_DATASET_TERMS` to `True` only if you accept those terms.
The cell sends a newline to the official downloader's confirmation prompt on your behalf.

The starter setting downloads at most 300 videos per category, across the real category and four manipulation methods.
Set `DOWNLOAD_LIMIT = 0` for all C23 videos only if you have sufficient bandwidth and temporary Colab storage (the authors estimate about 10 GB for full C23).
Free Drive is not used for these video files. Keep the official downloader's help visible if a server option needs changing.
If the downloader fails, stop and share the error rather than continuing with incomplete data.


In [ ]:
I_ACCEPT_DATASET_TERMS = False  # Set True only after reading and accepting the official agreement.
DOWNLOAD_LIMIT = 300
if not I_ACCEPT_DATASET_TERMS:
    raise RuntimeError('Read the official dataset agreement, then set I_ACCEPT_DATASET_TERMS = True if you accept.')
VIDEOS = Path('/content/ffpp_videos')
VIDEOS.mkdir(exist_ok=True)
for category in ['original', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']:
    command = [sys.executable, str(DOWNLOADER), str(VIDEOS), '-d', category, '-c', 'c23', '-t', 'videos']
    if DOWNLOAD_LIMIT > 0:
        command += ['--num_videos', str(DOWNLOAD_LIMIT)]
    print('Downloading', category, flush=True)
    subprocess.run(command, input='\n', text=True, check=True)
print('Downloads completed. Continue to face preparation.')


## 6. Prepare faces and cache them to Drive
This stage uses the official split lists, detects faces, saves crops and checks for data leakage.
The starter preparation caps videos separately within each split and method. The training selection is reproducible.
Inspect `preparation.json` for class counts and face-detection rejection rates.

Keep the same prepared data throughout one run. If you change the dataset after training starts, create a separate run folder rather than resuming incompatible weights.
This step can take time. Subsequent sessions can restore the prepared data archive instead.


In [ ]:
subprocess.run([sys.executable, 'prepare_ffpp.py', '--videos', '/content/ffpp_videos',
                '--output', 'data', '--train-videos', '120', '--eval-videos', '30',
                '--train-frames', '12', '--eval-frames', '16'], check=True)
import json
audit = json.loads((ROOT/'data'/'preparation.json').read_text())
print(json.dumps(audit, indent=2))
archive = shutil.make_archive('/content/face_data', 'zip', root_dir=ROOT/'data')
temporary_cache = SAVE/'face_data.uploading.zip'
shutil.copy2(archive, temporary_cache)
temporary_cache.replace(CACHE)
print('Prepared faces saved to Drive:', CACHE)


## 7. Train or resume the CNN + frequency detector
This is the main training step. A free T4 is recommended. The model downloads ImageNet initialization weights on first use.
Each completed epoch saves `last.pt` for resuming. `best.pt` is selected using validation balanced accuracy.

Start with 5 epochs to verify learning. After examining **validation** results, increase `EPOCHS` to 20 and rerun to resume.
If memory runs out, reduce `BATCH_SIZE` to 4 and set `ACCUMULATION` to 4. Stop using test data to guide tuning.
Do not leave a keep-alive script running; use Colab interactively and resume after ordinary interruptions.


In [ ]:
EPOCHS = 5  # Increase to 20 after the first run if validation learning and free GPU time permit.
BATCH_SIZE = 8
ACCUMULATION = 2
assert torch.cuda.is_available(), 'No free GPU is available. Save the notebook and retry later.'
assert (ROOT/'data'/'manifest.csv').is_file(), 'Complete or restore the prepared dataset first.'
command = [sys.executable, 'train.py', '--manifest', 'data/manifest.csv', '--output', str(RUN),
           '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--accumulation', str(ACCUMULATION)]
if (RUN/'last.pt').exists():
    command += ['--resume', str(RUN/'last.pt')]
subprocess.run(command, check=True)
print('Training finished. Selected checkpoint:', RUN/'best.pt')


## 8. Review learning on validation data
These plots are for training decisions. They are not the final test result.
If performance is poor, preserve the run and ask for help before changing many settings.


In [ ]:
import json
import matplotlib.pyplot as plt
history = json.loads((RUN/'history.json').read_text())
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
epochs = [x['epoch'] for x in history]
axes[0].plot(epochs, [x['train_loss'] for x in history], marker='o')
axes[0].set(title='Training loss', xlabel='Epoch')
axes[1].plot(epochs, [x['validation']['accuracy'] for x in history], label='Accuracy')
axes[1].plot(epochs, [x['validation']['balanced_accuracy'] for x in history], label='Balanced accuracy')
axes[1].set(title='Validation only', xlabel='Epoch', ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.show()


## 9. Calibrate confidence using validation data
Run after selecting your final training configuration. Calibration rescales confidence and does not manufacture a higher accuracy score.
If you train more afterward, run calibration again before making the final test evaluation.


In [ ]:
subprocess.run([sys.executable, 'evaluate.py', '--calibrate', '--manifest', 'data/manifest.csv',
                '--checkpoint', str(RUN/'best.pt'), '--output', str(RUN)], check=True)
print((RUN/'calibration.json').read_text())


## 10. Final held-out test evaluation
Run once your training choices are frozen. This creates metrics, per-image predictions, a confusion matrix and ROC plot.
If accuracy is below 95%, report it honestly. Test results must not be relabeled as validation results or used repeatedly for tuning.
The evaluation command refuses to overwrite an existing test report.


In [ ]:
subprocess.run([sys.executable, 'evaluate.py', '--manifest', 'data/manifest.csv',
                '--checkpoint', str(RUN/'calibrated.pt'), '--output', str(RUN/'evaluation')], check=True)
from IPython.display import display, Markdown
from PIL import Image
display(Markdown((RUN/'evaluation'/'RESULTS.md').read_text()))
display(Image.open(RUN/'evaluation'/'test_charts.png'))


## 11. Upload a face and inspect the prediction
This demo runs inside your notebook, without publishing a public application link.
Select a photograph with exactly one visible face. The trained checkpoint is required.
Red marks support a FAKE prediction; blue marks support a REAL prediction. The heatmap is model attention, not proof of edited pixels.
You may run this cell again to analyze another photograph.


In [ ]:
from google.colab import files
from deepfake.predict import Predictor
from PIL import Image
from IPython.display import display, Markdown
import io
predictor = Predictor(RUN/'calibrated.pt')
uploaded = files.upload()
for name, content in uploaded.items():
    try:
        result = predictor.predict(Image.open(io.BytesIO(content)))
        display(Markdown(f"## {result['prediction']} — {result['confidence']:.1%} model confidence"))
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(result['original']); axes[0].set_title('Original')
        axes[1].imshow(result['overlay']); axes[1].set_title('Facial Grad-CAM: ' + result['heatmap_class'])
        for axis in axes: axis.axis('off')
        plt.tight_layout(); plt.show()
        print(result['explanation'])
        print(result['confidence_type'] + ' confidence; performance on arbitrary new generators is unverified.')
    except (ValueError, OSError) as error:
        print('Could not analyze image:', error)


## 12. Export your trained application and results
This creates a project zip in Drive containing the source, your calibrated detector and measured results. It excludes the dataset and official downloader.
Download that zip from your Drive to run `app.py` locally following README.md. Copy the measured results into PROJECT_REPORT.md and include actual screenshots from step 11.
The report remains a draft until those experimental details are filled in.


In [ ]:
assert (RUN/'calibrated.pt').exists(), 'Train and calibrate before exporting a finished detector.'
assert (RUN/'evaluation'/'test_metrics.json').exists(), 'Complete the held-out evaluation before exporting the final project.'
import zipfile
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as source:
    output_zip = SAVE/'Trained_DeepFake_Project.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        for name in source.namelist():
            z.writestr('deepfake_project/'+name, source.read(name))
        z.write(RUN/'calibrated.pt', 'deepfake_project/runs/frequency/calibrated.pt')
        for name in ['history.json', 'calibration.json']:
            if (RUN/name).exists(): z.write(RUN/name, 'deepfake_project/runs/frequency/'+name)
        if (RUN/'evaluation').exists():
            for path in (RUN/'evaluation').rglob('*'):
                if path.is_file():
                    z.write(path, 'deepfake_project/runs/frequency/evaluation/'+str(path.relative_to(RUN/'evaluation')))
print('Your trained project:', output_zip)
print('Download this file from Google Drive. It does not contain the training dataset.')


## Optional: CNN-only comparison (if free GPU time remains)
Use exactly the same manifest. Train in a separate folder with `--mode cnn`; do not overwrite the frequency model.
Calibration and evaluation must also use the separate checkpoint and output folder. Report both models even if the frequency model does not improve the measured score.


In [ ]:
RUN_CNN_BASELINE = False
if RUN_CNN_BASELINE:
    BASELINE = SAVE/'runs'/'cnn'
    command = [sys.executable, 'train.py', '--manifest', 'data/manifest.csv', '--mode', 'cnn',
               '--output', str(BASELINE), '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
               '--accumulation', str(ACCUMULATION)]
    if (BASELINE/'last.pt').exists(): command += ['--resume', str(BASELINE/'last.pt')]
    subprocess.run(command, check=True)
    subprocess.run([sys.executable, 'evaluate.py', '--calibrate', '--checkpoint', str(BASELINE/'best.pt'),
                    '--output', str(BASELINE)], check=True)
    subprocess.run([sys.executable, 'evaluate.py', '--checkpoint', str(BASELINE/'calibrated.pt'),
                    '--output', str(BASELINE/'evaluation')], check=True)
